# Generate frequencies.txt

Generates GTFS frequencies file with headway intervals for each route.

In [1]:
import json
from pathlib import Path 
import pandas as pd

## Parameters

In [2]:
# Path to params.json (same directory as this notebook)
_params_path = "../params.json"
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

In [3]:
CITY = p["city"]

# Load headways configured by route
headway_by_route = p["frequencies"].get("headway_by_route", {})
interval_global = headway_by_route.get("default", 13.11)

print(f"Global headway: {interval_global} minutes")
print(f"Routes with specific headway: {len([k for k in headway_by_route.keys() if k != 'default'])}")

params_frequencies = {
    "start_time": p["frequencies"]["start_time"],
    "end_time": p["frequencies"]["end_time"],
    "headway_secs": round(interval_global * 60, 2),
    "exact_times": p["frequencies"]["exact_times"],
}

Global headway: 13.11 minutes
Routes with specific headway: 5


In [4]:
# --- GTFS Folder ---
PATH_DIR_GTFS = Path(f"../data/{CITY}/gtfs-frequencies")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_GTFS.absolute()}")

# --- Processed Folder ---
PATH_DIR_processed = Path(f"../data/{CITY}/processed")
PATH_DIR_processed.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_processed.absolute()}")

Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/gtfs-frequencies
Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/processed


In [5]:
# params_frequencies loaded from params.json in the previous cell

In [6]:
# --- GTFS Folder ---
PATH_DIR_GTFS = Path(f"../data/{CITY}/gtfs-frequencies")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_GTFS.absolute()}")

# --- Processed Folder ---
PATH_DIR_processed = Path(f"../data/{CITY}/processed")
PATH_DIR_processed.mkdir(parents=True, exist_ok=True)
print(f"Output: {PATH_DIR_processed.absolute()}")

Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/gtfs-frequencies
Output: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generacionDatos/generation-gtfs-from-shapes/1-gtfs-generation/../data/merida/processed


## Read files

In [7]:
stop_times = pd.read_csv(PATH_DIR_GTFS / "stop_times.txt")

## Load routes for name to ID mapping
routes_df = pd.read_csv(PATH_DIR_GTFS / "routes.txt")

# Create mapping from route_long_name to route_name
name_to_id = dict(zip(routes_df["route_long_name"], routes_df["route_name"]))

# Convert configuration by route name to route_name
headway_by_route_name = {}
for route_name, headway in headway_by_route.items():
    if route_name == "default":
        headway_by_route_name["default"] = headway
    elif route_name in name_to_id:
        headway_by_route_name[name_to_id[route_name]] = headway
        print(f"Mapping '{route_name}' -> {name_to_id[route_name]}: {headway} min")
    else:
        print(f"Warning: Route '{route_name}' not found in routes.txt")

# Function to get headway by route
def get_headway_for_route(route_name, default_headway):
    return headway_by_route_name.get(route_name, default_headway)

print(f"Specific headways configured for {len([k for k in headway_by_route_name.keys() if k != 'default'])} routes")

stop_times.head()

Specific headways configured for 0 routes


,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time
0,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0000,1,00:00:00,00:00:12
1,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0001,2,00:00:47,00:00:59
2,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0002,3,00:01:34,00:01:46
3,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0003,4,00:02:22,00:02:34
4,100_52 Norte Villas La Hacienda R-1_trip_00,1,100_52 Norte Villas La Hacienda R-1_0004,5,00:03:09,00:03:21


## Generate frequencies.txt

# Convert configuration by route name to route_name
headway_by_route_name = {}
for route_name, headway in headway_by_route.items():
    if route_name == "default":
        headway_by_route_name["default"] = headway
    elif route_name in name_to_id:
        headway_by_route_name[name_to_id[route_name]] = headway

In [8]:
# Basic validations
def _valid_hhmmss(s):
    try:
        h,m,sec = map(int, str(s).split(":"))
        return (h >= 0 and 0 <= m < 60 and 0 <= sec < 60)
    except Exception:
        return False


good_start_bool = _valid_hhmmss(params_frequencies["start_time"])
good_end_bool   = _valid_hhmmss(params_frequencies["end_time"])

# If 'not' start_time IS VALID 'or' 'not' end_time IS VALID:
if not (good_start_bool and good_end_bool):
    raise ValueError("Invalid time format in start_time/end_time (use HH:MM:SS; hours >=24 allowed).")

In [9]:
def build_frequencies(stop_times: pd.DataFrame, dic_params: list):
    """
    Generate frequencies.txt from stop_times.
    """
    # Get unique trips
    trips = stop_times[["trip_id"]].drop_duplicates().reset_index(drop=True)

    # Convert parameter list to DataFrame
    dic_params_df = pd.DataFrame(dic_params, index=[0])

    # Ensure exact_times exists
    if "exact_times" not in dic_params_df.columns:
        dic_params_df["exact_times"] = 0
    
    # Cartesian product (Cross Join)
    # In modern Pandas versions (1.2+), you can use how="cross"
    frequencies = trips.merge(dic_params_df, how="cross")

    # Clean types and standard GTFS column order
    cols_order = ["trip_id", "start_time", "end_time", "headway_secs", "exact_times"]
    frequencies = frequencies[cols_order].copy()
    
    frequencies["headway_secs"] = frequencies["headway_secs"].astype(int)
    frequencies["exact_times"]  = frequencies["exact_times"].astype(int)

    return frequencies

def build_frequencies_by_route(stop_times: pd.DataFrame, headway_config: dict, global_params: dict):
    """
    Generate frequencies.txt with specific headways by route.
    """
    frequencies_rows = []
    
    # Get unique trips with their route_names
    trips_unique = stop_times[["trip_id"]].drop_duplicates()
    
    # Extract route_name from trip_id (format: "Route_X_trip_00")
    trips_unique["route_name"] = trips_unique["trip_id"].str.extract(r'(Route_\\d+)')
    
    for _, trip_row in trips_unique.iterrows():
        trip_id = trip_row["trip_id"]
        route_name = trip_row["route_name"]
        
        # Get specific headway or default
        headway_mins = get_headway_for_route(route_name, global_params["interval_global"])
        headway_secs = round(headway_mins * 60, 2)
        
        # Create frequency record
        freq_row = {
            "trip_id": trip_id,
            "start_time": global_params["start_time"],
            "end_time": global_params["end_time"],
            "headway_secs": int(headway_secs),
            "exact_times": global_params["exact_times"]
        }
        frequencies_rows.append(freq_row)
    
    return pd.DataFrame(frequencies_rows)

In [10]:
global_params = {
    "start_time": p["frequencies"]["start_time"],
    "end_time": p["frequencies"]["end_time"],
    "interval_global": interval_global,
    "exact_times": p["frequencies"]["exact_times"]
}

if headway_by_route_name and any(k != "default" for k in headway_by_route_name.keys()):
    print("Generating frequencies with specific headways by route...")
    frequencies = build_frequencies_by_route(stop_times, headway_by_route_name, global_params)
else:
    print("Generating frequencies with global headway...")
    frequencies = build_frequencies(stop_times, dic_params=params_frequencies)

# Show headway summary by route
print("Headway summary by route:")
headway_summary = frequencies.copy()
headway_summary["route_name"] = headway_summary["trip_id"].str.extract(r'(Route_\d+)')
headway_summary = headway_summary.groupby("route_name")["headway_secs"].first().reset_index()
headway_summary["headway_min"] = (headway_summary["headway_secs"] / 60).round(2)
print(headway_summary.head(10))

# Reorder columns to GTFS standard order
frequencies = frequencies[["trip_id", "start_time", "end_time", "headway_secs", "exact_times"]]
frequencies.head()

Generating frequencies with global headway...
Headway summary by route:
Empty DataFrame
Columns: [route_name, headway_secs, headway_min]
Index: []


,trip_id,start_time,end_time,headway_secs,exact_times
0,100_52 Norte Villas La Hacienda R-1_trip_00,06:00:00,07:00:00,786,1
1,112_Mulsay Juan Pablo Ii_trip_00,06:00:00,07:00:00,786,1
2,114_Carranza_trip_00,06:00:00,07:00:00,786,1
3,115_San Lucas_trip_00,06:00:00,07:00:00,786,1
4,11_50 Penal Paso Texas_trip_00,06:00:00,07:00:00,786,1


## Export

In [11]:
# Export frequencies to GTFS format
frequencies.to_csv(PATH_DIR_GTFS / "frequencies.txt", index=False)
print(f"✓ Exported {len(frequencies)} frequency records to frequencies.txt")

✓ Exported 56 frequency records to frequencies.txt
